# 00 — Audit dữ liệu & chốt thước đo

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/00_data_audit.ipynb)

**Không cần GPU.** Runtime → CPU. Khoảng 1 phút.

Đây là nấc 0 của lộ trình: trước khi chạy bất cứ thí nghiệm nào, phải chắc rằng **thước đo
đúng**. Nếu executor sai thì mọi con số PA/EA ở 5 nấc sau đều vô nghĩa, và tệ hơn — ở nấc 3
và nấc 5, model sẽ *học* từ tín hiệu sai.

## Lộ trình

| Nấc | Kỹ thuật | Notebook |
|:---:|---|---|
| — | Audit dữ liệu & chốt thước đo | `00_data_audit.ipynb` |
| 1 | Inference thông thường | `01_baseline_basic.ipynb` |
| 2 | + Prompt engineering | `02_prompt_engineering.ipynb` |
| 3 | + SFT trên Qwen3-8B | `03_sft_qwen3.ipynb` |
| 4 | + Self-evaluation 2 bước | `04_self_evaluation.ipynb` |
| 5 | + ACE (playbook + truy hồi) | `05_ace.ipynb` |
| — | Tổng hợp & kiểm định | `06_final_report.ipynb` |

## Notebook này làm gì

1. Kiểm chứng executor DSL bằng cách bắt nó tái tạo `exe_ans` từ chính gold program.
2. Thống kê dữ liệu: nguồn, độ phức tạp, mật độ `table_*`.
3. Đo nhiễu nhãn vàng — lần chạy trước chỉ mô tả định tính.
4. Chốt hai chỉ số dùng xuyên suốt: `PA_loose` (so được với mốc tham chiếu) và `EA`.

## §1. Môi trường

In [ ]:
%%capture
!pip install -q pandas matplotlib

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, stats

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

## §2. Kiểm chứng executor

Nhãn vàng là chuẩn không thể bàn cãi: executor nào không tái tạo được `exe_ans` từ chính
gold program thì executor đó sai, bất kể model dự đoán ra sao.

Hai điểm khác biệt quan trọng so với FinQA mà executor phải xử lý đúng:

* `table_*(nhãn, none)` đọc theo **nhãn hàng** (cột đầu tiên), không phải tên cột;
* ViNumQA **không dùng** `const_*`, và có toán hạng dạng `20%` nghĩa là 0.2.

In [ ]:
print(f"{'tập':<8}{'n':>6}{'khớp exe_ans':>16}{'tỉ lệ':>9}{'gold có table_*':>18}{'table_* đúng':>15}")
print("-" * 74)
for name, samples in [("train", train_all), ("valid", valid_all), ("test", test_all)]:
    ok = tbl_n = tbl_ok = 0
    for s in samples:
        v = dsl.execute_program(s["qa"]["program"], s.get("table") or [])
        good = dsl.check_ea(v, s["qa"].get("exe_ans"))
        ok += good
        if "table_" in s["qa"]["program"]:
            tbl_n += 1; tbl_ok += good
    print(f"{name:<8}{len(samples):>6}{ok:>16}{ok/len(samples):>9.2%}"
          f"{tbl_n:>18}{f'{tbl_ok}/{tbl_n}':>15}")

assert all(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                        s["qa"].get("exe_ans")) for s in test_all), \
    "Có gold trên test không tái tạo được exe_ans — DỪNG LẠI."
print("\n✅ Executor tái tạo đúng 100% nhãn vàng trên test → thước đo dùng được.")

## §3. Dữ liệu có gì

ViNumQA ghép hai nguồn, tách được từ `id`:

* **FinQA-Vi** — dịch từ FinQA, giữ id gốc dạng `HIG/2004/page_140.pdf-3`
* **Vi Data** — báo cáo doanh nghiệp Việt Nam 2020–2025

Hai nguồn khác nhau rõ rệt về cách dùng `table_*`, nên model cần chiến lược khác nhau —
điều này sẽ quay lại ở nấc 5 khi ACE học playbook.

In [ ]:
print(f"{'tập':<8}{'nguồn':<12}{'n':>6}{'table_*':>10}{'≥3 bước':>10}  phân bố số phép")
print("-" * 78)
for name, samples in [("train", train_all), ("valid", valid_all), ("test", test_all)]:
    for src, sub in sorted(data.split_by_source(samples).items()):
        tbl = sum(1 for s in sub if "table_" in s["qa"]["program"])
        hard = sum(1 for s in sub if dsl.n_ops(s["qa"]["program"]) >= 3)
        dist = dict(sorted(Counter(min(dsl.n_ops(s["qa"]["program"]), 5)
                                   for s in sub).items()))
        print(f"{name:<8}{src:<12}{len(sub):>6}{tbl:>10}{hard:>10}  {dist}")

## §4. Nhiễu nhãn vàng

Lần chạy trước chỉ nêu định tính "5–10 % gold program có vấn đề". Ở đây định lượng hai loại
**đo được bằng máy**:

1. **`multiply(#n, 100)`** — prompt của dự án *cấm* dạng này (quy tắc "tỷ lệ trả về số thập
   phân"), nên trên các mẫu đó model tuân thủ prompt vẫn bị chấm sai **cả PA lẫn EA**.
2. **Gold không tự thực thi ra `exe_ans`** — thường do thiếu dấu đóng ngoặc.

Loại thứ nhất là loại nguy hiểm, và điều đáng chú ý nhất là **nó nằm gần như toàn bộ ở
train** — đúng tập mà nấc 3 (SFT) và nấc 5 (ACE) dùng để học.

In [ ]:
audits = {}
for name, samples in [("train", train_all), ("valid", valid_all), ("test", test_all)]:
    audits[name] = data.audit_gold(samples, name)
    data.print_audit(audits[name]); print()

print(f"{'═'*70}\n  NHIỄU NHÃN THEO TẬP\n{'═'*70}")
print(f"{'tập':<8}{'n':>6}{'multiply(100)':>16}{'gold lỗi cú pháp':>19}{'tổng':>8}{'%':>8}")
for name, r in audits.items():
    print(f"{name:<8}{r['n']:>6}{r['multiply_100']:>16}{r['gold_not_executable']:>19}"
          f"{r['noisy']:>8}{r['noisy_pct']:>8.2f}")

print(f"\n  → Nhiễu tập trung ở TRAIN, valid/test gần như sạch.")
print(f"  → Nấc 3 (SFT) và nấc 5 (ACE) đều học từ train nên phải lọc các mẫu này;")
print(f"    hai notebook đó dùng data.is_noisy_gold() để loại chúng.")

noisy_path = os.path.join(RESULT_DIR, "noisy_train_ids.json")
with open(noisy_path, "w", encoding="utf-8") as f:
    json.dump({"all_noisy": audits["train"]["noisy_ids"],
               "multiply_100": audits["train"]["multiply_100_ids"],
               "gold_not_executable": audits["train"]["gold_not_executable_ids"]},
              f, ensure_ascii=False, indent=1)
print(f"\n[SAVE] danh sách id nhiễu → {noisy_path}")

## §5. Hai chỉ số dùng xuyên suốt

| Chỉ số | Định nghĩa | Dùng để |
|---|---|---|
| **EA** | thực thi program dự đoán, so với `exe_ans` (làm tròn 5 chữ số) | chỉ số chính |
| **PA_strict** | chuẩn hoá canonical: giao hoán `add`/`multiply`, `100.00`≡`100`, `20%`≡`0.2`, kiểm tra `#N` hợp lệ | so sánh giữa các nấc |
| **PA_loose** | đúng công thức `normalize_program` của repo cũ | **so trực tiếp với bảng tham chiếu** |

Cell dưới kiểm chứng rằng `PA_loose` tái lập đúng con số tham chiếu — bằng chứng rằng
thước đo mới không làm lệch so với mốc tham chiếu.

In [ ]:
_pub = os.path.join(REPO_DIR, "reference", "baseline_results", "Qwen3-8B_program.csv")
if os.path.exists(_pub):
    _rows = io_utils.score_saved_predictions(_pub, test_all, "Qwen3-8B tham chiếu")
    _m = pipeline.summarize(_rows, "tham chiếu: Qwen3-8B + self-eval")
    _ref = io_utils.BASELINE_RESULTS["Qwen3-8B"]
    print(f"  PA tham chiếu    : {_ref['PA']:.2f}%")
    print(f"  PA_loose chấm lại: {_m['PA_loose']*100:.2f}%   "
          f"{'✅ khớp' if abs(_m['PA_loose']*100 - _ref['PA']) < 0.1 else '⚠ lệch'}")
    print(f"  EA tham chiếu    : {_ref['EA']:.2f}%")
    print(f"  EA chấm lại      : {_m['EA']*100:.2f}%")
    print(f"\n  Chênh lệch EA đến từ câu dùng table_*: executor nội bộ khi viết bài bỏ sót")
    print(f"  một phần các câu này. Xem reference/README.md.")
    print(f"\n  → Mốc tham chiếu cho cả lộ trình: Qwen3-8B + self-eval")
    print(f"    đạt PA {_ref['PA']:.2f}% / EA {_m['EA']*100:.2f}% (EA đã chấm lại).")
else:
    print("(không thấy reference/baseline_results/ — bỏ qua phần đối chiếu)")

## Kết luận nấc 0

Thước đo đã chốt. Từ đây mọi notebook đều:

* chấm bằng `vinumqa.dsl` (executor vừa kiểm chứng ở §2);
* báo cáo đồng thời `EA`, `PA_strict`, `PA_loose`;
* lưu kết quả vào `RESULT_DIR/<tên nấc>.jsonl` để notebook sau ghép cặp và kiểm định.

**Tiếp theo:** `01_baseline_basic.ipynb` — nấc 1, inference thông thường.